# R02 causal raw factors on mini_7

**Question:** Can the authorized development excerpt enter a receiver-like causal measurement path and yield physically defined D counts and conditional G novelty without evaluator information?

This is one contiguous development excerpt, not independent scenes. The recorded source stamps use integer nanoseconds; sync indices are association IDs. Arrival equals source by explicit replay assumption, not measured network telemetry. Raw XYZ are metres; D is points per 45° × radial cell per frame; G is directed 90th percentile nearest centroid distance in metres. No reference, calibration, score, attack or outcome is used.

In [ ]:
import os, sys, csv, json
from pathlib import Path
ROOT=Path.cwd(); sys.path.insert(0,str(ROOT/'src'))
RUN=Path(os.environ['VP_RUN_DIR']); cfg=json.loads((RUN/'config.json').read_text())
print('run:',RUN.name,'\nsource:',cfg['dataset'],cfg['revision'],cfg['segment'])
print('receipt:',cfg['receipt_model'])
print('geometry:',cfg['range_edges_m'],'m; sectors:',cfg['azimuth_sectors'],'; voxel:',cfg['voxel_m'],'m; cap:',cfg['voxel_cap'])
print('reference/calibration:',cfg['reference_id'],cfg['calibration_id'])

In [ ]:
from valid_point.replay import AllowedSource, ReplayConfig
from valid_point.operational import measure_operational
raw=ROOT/'data/mixed_signals'/cfg['revision']/'raw'
source=AllowedSource(raw/'PointClouds/mini_7',raw/'Odometry/mini_7')
events=source.events(max_sync_id=1,first_seconds=cfg['first_seconds'])
poses={s:source.read_poses(s) for s in ('003','004','laser')}
records=measure_operational(events,source,config=ReplayConfig(min_voxels=cfg['min_voxels']),poses=poses)
for rec in records:
 r=rec.raw
 print(r.stream,r.principal,'sync',r.sync_id,'source ns',r.source_ns,'pose',r.pose_id,'history',r.history_id,
       'D',r.density.status, 'counts',r.density.counts,'outside',r.density.excluded_points,
       'G m',r.geometry.distance_m,'G reason',r.geometry.reason)

In [ ]:
def read_table(name):
 with (RUN/(name+'.csv')).open(newline='') as f: return list(csv.DictReader(f))
inputs=read_table('T_R02_inputs'); eligibility=read_table('T_R02_eligibility')
components=read_table('T_R02_components'); timeline=read_table('F_R02_raw_timelines')
overlap=read_table('T_R02_overlap_diagnostic')
print('Full table rows:',{k:len(v) for k,v in [('inputs',inputs),('eligibility',eligibility),('components',components),('timeline',timeline)]})
print('Sensor/principal, D and G coverage:')
for row in components: print(row)
print('Bounded top-versus-vehicle potential overlap/skew (diagnostic only):')
for row in overlap: print(row)
print('Selected pose/history IDs and unknowns (first 15):')
for a,b in list(zip(inputs,eligibility))[:15]: print(a['cloud_id'],a['principal'],a['pose_id'],a['history_id'],b['G_status'],b['G_reason'])

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display,Image
for filename in ('F_R02_raw_timelines.png','F_R02_alignment.png'):
 display(Image(filename=str(RUN/filename)))
print('Figures show all selected streams; absent numeric G is a gap, not zero.')

In [ ]:
env=json.loads((RUN/'environment.json').read_text()) if (RUN/'environment.json').exists() else None
isolation=json.loads((RUN/'boundary_isolation.json').read_text())
print('Boundary isolation:',{k:v for k,v in isolation.items() if k!='access_log'})
print('Accessed files:',len(isolation['access_log']),'all cloud paths:',all('/PointClouds/mini_7/' in p for p in isolation['access_log']))
print('Tests:',(RUN/'test.log').read_text()[-1000:])
if env: print('CPU/RSS:',env)
print('Technical gate is separate from science; no efficacy or false-alarm claim is made.')